# Combined Agent: UC-First + Genie Fallback + Lakebase Memory + Parallel MCP

This notebook combines the best features from both agent architectures:

## Architecture

```
User Query → UC Functions (Parallel MCP) → Sufficiency Check → [If sufficient] → Response
                                                             ↓
                                                     [If partial/insufficient]
                                                             ↓
                                              Genie Fallback (unanswered parts only)
                                                             ↓
                                                   Synthesize → Response
```

## Features

### From UC-First Genie-Fallback:
- **UC functions tried FIRST** - deterministic, fast
- **Partial answer detection** - identifies what was/wasn't answered
- **Targeted Genie queries** - only asks about unanswered parts
- **Intelligent synthesis** - combines both responses seamlessly

### From Lakebase + MCP Agent:
- **Parallel tool execution** via MCP - all tools run simultaneously
- **Lakebase PostgreSQL memory** - conversation persistence across sessions
- **Connection pooling** - efficient database connections
- **OAuth credential caching** - 50-minute token cache

## Prerequisites

1. Run `00_setup` notebook first to create `config/atbat_assistant.json`
2. Ensure `genie` and `lakebase` sections are in the config
3. Lakebase instance configured (for memory)

In [ ]:
%pip install -U -qqqq backoff databricks-openai uv databricks-agents "mlflow>=3.9" databricks-mcp langgraph-checkpoint-postgres psycopg[binary,pool] databricks-langchain langgraph
dbutils.library.restartPython()

## Load Configuration

Load configuration from `config/atbat_assistant.json` (created by setup notebook).

In [ ]:
# Load configuration from setup notebook
import json
from pathlib import Path
import mlflow

CONFIG = json.loads(Path("config/atbat_assistant.json").read_text())

# Extract configuration variables
PROMPT_NAME = CONFIG["prompt_registry"]["prompt_name"]
LLM_ENDPOINT_NAME = CONFIG["llm"]["endpoint_name"]
UC_MODEL_NAME = CONFIG["model"]["uc_model_name"]
MODEL_NAME = CONFIG["model"]["model_name"]
UC_TOOL_NAMES = CONFIG["tools"]["uc_tool_names"]
CATALOG = CONFIG["workspace"]["catalog"]
SCHEMA = CONFIG["workspace"]["schema"]

# Genie config
GENIE_SPACE_ID = CONFIG["genie"]["space_id"]
GENIE_NAME = CONFIG["genie"]["name"]

# Lakebase config
LAKEBASE_INSTANCE = CONFIG["lakebase"]["instance_name"]
LAKEBASE_HOST = CONFIG["lakebase"]["host"]

# Set MLflow experiment
EXPERIMENT_ID = CONFIG["mlflow"]["experiment_id"]
mlflow.set_experiment(experiment_id=EXPERIMENT_ID)

print(f"Loaded config from: config/atbat_assistant.json")
print(f"LLM Endpoint: {LLM_ENDPOINT_NAME}")
print(f"UC Tools: {len(UC_TOOL_NAMES)} functions")
print(f"Genie Space: {GENIE_SPACE_ID}")
print(f"Lakebase Host: {LAKEBASE_HOST or '(not configured - memory disabled)'}")

## Register System Prompt

The prompt includes routing logic that prioritizes UC Functions for precise tasks
and falls back to the Genie Space for broader data exploration.

In [ ]:
# Define the prompt template
PROMPT_TEMPLATE = """You are a hitting assistant tasked with helping batters prepare for matchups against specific pitchers. 

Note that the output from the embedding vectors for pitchers and batters reflects the minmax scaled value for each feature included with the following schemas. This information is provided so you can interpret the embeddings directly for analysis:

Pitcher:release_speed,release_spin_rate,release_pos_x,release_pos_y,release_pos_z,release_extension,pfx_x,pfx_z,vx0,vy0,vz0,ax,ay,az,effective_speed,arm_angle 

Batter:launch_speed,launch_angle,hit_distance_sc,estimated_ba_using_speedangle,estimated_woba_using_speedangle,woba_value,woba_denom,babip_value,iso_value,launch_speed_angle,barrel

The team abbreviations to choose from are below, use the 3 letter acronyms to get data:

Teams:TEX,CHC,LAA,LAD,STL,PHI,ARI,OAK,TBR,MIN,CLE,CHW,NYM,COL,SEA,MIA,SDP,WSN,HOU,SFG,CIN,BAL,KCR,PIT,ATL,NYY,DET,MIL,TOR,BOS,ATH

General rules:

- Always assume the most recent season (2025) if a season is not provided. 
- Always leverage the tooling you have available to answer user queries.
- Only perform the minimum necessary tool calls to complete a request. Do not exceed 8 tool calls before providing a response.
- If you need multiple tools, include all tool_calls in a single assistant message; don't chain them one-by-one.

    - For open-ended requests always respond with the following format in markdown:
    # At-Bat Assistant Assessment
    ## Data collected 
    - Summarize the data collected to inform the analysis in a very concise fashion. You do not need reference the tools by their explicit name, just need to summarize the data collected. Ex. Collected data on tendencies by count. Do not exceed 50 words
    ## Pitcher Approach
    - Summarize how the pitcher might approach the batter. Use discretion to include further subheadings by count or scenario, or just include it all under the pitcher approach heading if that is not necessary. Do not exceed 200 words
    ## Recommendation
    - Summarize how the batter should approach potential at-bats(s) in a concise format, 50-75 words.

CONVERSATION HISTORY: You have access to previous messages in this conversation thread.
- ONLY reference prior conversation context if the user's current question is ambiguous or explicitly refers to something discussed earlier.
- If the user asks a NEW, self-contained question, answer it directly using your tools WITHOUT referencing prior context.
- Do NOT proactively bring up previous topics or assume the user wants comparisons to earlier queries.
"""

# Register prompt if it doesn't exist OR if the template has changed
try:
    existing_prompt = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}@production")
    if existing_prompt.template == PROMPT_TEMPLATE:
        system_prompt = existing_prompt
        print(f"Prompt '{PROMPT_NAME}' is up-to-date (version {system_prompt.version}, @production)")
    else:
        print(f"Prompt '{PROMPT_NAME}' template has changed. Registering new version...")
        system_prompt = mlflow.genai.register_prompt(
            name=PROMPT_NAME,
            template=PROMPT_TEMPLATE,
            commit_message="Update prompt with tool parameter rules",
        )
        mlflow.genai.set_prompt_alias(
            name=PROMPT_NAME,
            alias="production",
            version=system_prompt.version,
        )
        print(f"Registered version {system_prompt.version} and set @production alias")
except Exception:
    # Prompt doesn't exist or has no @production alias — register it
    print(f"Prompt '{PROMPT_NAME}' not found. Registering...")
    system_prompt = mlflow.genai.register_prompt(
        name=PROMPT_NAME,
        template=PROMPT_TEMPLATE,
        commit_message="Initial version of at-bat assistant prompt",
    )
    print(f"Registered prompt '{PROMPT_NAME}' (version {system_prompt.version})")

    # Set @production alias to the newly created version
    mlflow.genai.set_prompt_alias(
        name=PROMPT_NAME,
        alias="production",
        version=system_prompt.version,
    )
    print(f"Set @production alias -> version {system_prompt.version}")

## Setup Lakebase (Optional)

If you want conversation memory, set up Lakebase. Skip this cell if you don't need memory.

In [ ]:
# Only run if you have Lakebase configured and need to set up checkpoint tables
SETUP_LAKEBASE = True  # Set to True to create checkpoint tables
CHECKPOINT_SCHEMA = "checkpoint"  # Custom schema we own

if SETUP_LAKEBASE:
    from langgraph.checkpoint.postgres import PostgresSaver
    import psycopg
    import uuid

    from databricks.sdk import WorkspaceClient
    w = WorkspaceClient()

    current_user = w.current_user.me().user_name

    cred = w.database.generate_database_credential(
        request_id=str(uuid.uuid4()),
        instance_names=[LAKEBASE_INSTANCE],
    )

    conn = psycopg.connect(
        f"dbname=databricks_postgres user={current_user} host={LAKEBASE_HOST} sslmode=require",
        password=cred.token
    )
    conn.autocommit = True

    # Create a schema we own
    conn.execute(f'CREATE SCHEMA IF NOT EXISTS {CHECKPOINT_SCHEMA}')
    conn.execute(f'SET search_path TO {CHECKPOINT_SCHEMA}')
    print(f"Created and switched to schema: {CHECKPOINT_SCHEMA}")

    # Grant SP privileges on the new schema
    SP_CLIENT_ID = dbutils.secrets.get(scope=CONFIG["prompt_registry_auth"]["secret_scope_name"],
                                        key=CONFIG["prompt_registry_auth"]["oauth_client_id_key"])
    conn.execute(f'GRANT ALL ON SCHEMA {CHECKPOINT_SCHEMA} TO "{SP_CLIENT_ID}"')
    print(f"Granted schema privileges to SP: {SP_CLIENT_ID}")

    checkpointer = PostgresSaver(conn)
    checkpointer.setup()
    print("Lakebase checkpoint tables created!")

    # Grant SP access to the checkpoint tables
    conn.execute(f'GRANT ALL ON ALL TABLES IN SCHEMA {CHECKPOINT_SCHEMA} TO "{SP_CLIENT_ID}"')
    print(f"Granted table privileges to SP")

    conn.close()
    print(f"\nIMPORTANT: The agent's Lakebase connection must use search_path={CHECKPOINT_SCHEMA}")
else:
    print("Skipping Lakebase setup (set SETUP_LAKEBASE = True to create checkpoint tables)")

## Define the Agent Code

Below we define the combined agent code in a single cell, enabling us to write it to a local Python file using the `%%writefile` magic command for subsequent logging and deployment.

### Agent Features:
- **UC-first, Genie-fallback architecture** - Deterministic tools tried first
- **Parallel MCP tool execution** - All tool calls run simultaneously  
- **Partial answer detection** - Identifies what was/wasn't answered
- **Lakebase memory** - PostgreSQL-backed conversation persistence
- **Connection pooling** - Efficient database connections with OAuth caching

In [ ]:
%%writefile agent.py
"""
Combined Agent: UC-First with Genie Fallback + Lakebase Memory + Parallel MCP Tool Calling

This agent combines:
1. UC functions tried FIRST via parallel MCP execution (deterministic, fast)
2. Sufficiency evaluation with partial answer detection
3. Genie fallback for unanswered parts (flexible, novel queries)
4. Lakebase PostgreSQL memory for conversation persistence
5. Connection pooling and OAuth credential caching
"""

import asyncio
import json
import logging
import os
import time
import uuid
from concurrent.futures import ThreadPoolExecutor, as_completed
from contextlib import contextmanager
from pathlib import Path
from threading import Lock
from typing import Annotated, Any, Generator, Literal, Optional, Sequence, TypedDict

import mlflow
import psycopg
from databricks.sdk import WorkspaceClient
from databricks.sdk.config import Config
from databricks_langchain import ChatDatabricks, UCFunctionToolkit
from databricks_langchain.genie import GenieAgent
from databricks_mcp import DatabricksMCPClient
from langchain_core.messages import AIMessage, AIMessageChunk, BaseMessage, HumanMessage, ToolMessage
from langchain_core.runnables import RunnableConfig, RunnableLambda
from langgraph.checkpoint.postgres import PostgresSaver
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages
from mlflow.entities import SpanType
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import (
    ResponsesAgentRequest,
    ResponsesAgentResponse,
    ResponsesAgentStreamEvent,
)
from psycopg.rows import dict_row
from psycopg_pool import ConnectionPool
from pydantic import BaseModel

logger = logging.getLogger(__name__)

########################################
# UTILITIES
########################################

def get_dbutils():
    """Get dbutils for secrets access."""
    try:
        from pyspark.dbutils import DBUtils
        from pyspark.sql import SparkSession
        spark = SparkSession.builder.getOrCreate()
        return DBUtils(spark)
    except (ImportError, Exception):
        try:
            import IPython
            ipython = IPython.get_ipython()
            if ipython and "dbutils" in ipython.user_ns:
                return ipython.user_ns["dbutils"]
        except:
            pass
    return None

dbutils = get_dbutils()

########################################
# CONFIGURATION
########################################

_CONFIG_PATH = Path("config/atbat_assistant.json")
if _CONFIG_PATH.exists():
    CONFIG = json.loads(_CONFIG_PATH.read_text())
else:
    config_env = os.getenv("ATBAT_ASSISTANT_CONFIG_JSON")
    if config_env:
        CONFIG = json.loads(config_env)
    else:
        raise FileNotFoundError(
            "config/atbat_assistant.json not found and ATBAT_ASSISTANT_CONFIG_JSON env var is not set"
        )

# Extract configuration values
PROMPT_NAME = CONFIG["prompt_registry"]["prompt_name"]
LLM_ENDPOINT_NAME = CONFIG["llm"]["endpoint_name"]
UC_TOOL_NAMES = CONFIG["tools"]["uc_tool_names"]
CATALOG = CONFIG["workspace"]["catalog"]
SCHEMA = CONFIG["workspace"]["schema"]

# Genie configuration
GENIE_SPACE_ID = CONFIG["genie"]["space_id"]
GENIE_NAME = CONFIG["genie"]["name"]

# Auth configuration
SECRET_SCOPE_NAME = CONFIG["prompt_registry_auth"]["secret_scope_name"]
CLIENT_ID_KEY = CONFIG["prompt_registry_auth"]["oauth_client_id_key"]
CLIENT_SECRET_KEY = CONFIG["prompt_registry_auth"]["oauth_client_secret_key"]
DATABRICKS_HOST = CONFIG["prompt_registry_auth"]["databricks_host"]

# Set DATABRICKS_HOST if configured
if DATABRICKS_HOST and not os.getenv("DATABRICKS_HOST"):
    os.environ["DATABRICKS_HOST"] = DATABRICKS_HOST.rstrip("/")

# Load OAuth credentials from secrets into LOCAL variables (not env vars).
# The notebook runtime already sets DATABRICKS_TOKEN in the environment;
# putting OAuth creds there too causes a dual-auth conflict in the SDK.
_SP_CLIENT_ID = None
_SP_CLIENT_SECRET = None
if dbutils:
    try:
        _SP_CLIENT_ID = dbutils.secrets.get(scope=SECRET_SCOPE_NAME, key=CLIENT_ID_KEY).strip()
        _SP_CLIENT_SECRET = dbutils.secrets.get(scope=SECRET_SCOPE_NAME, key=CLIENT_SECRET_KEY).strip()
    except:
        pass

# Create WorkspaceClient: use OAuth M2M if available, otherwise auto-detect from env.
# Explicit auth_type prevents the SDK from discovering DATABRICKS_TOKEN in the
# environment and raising a dual-auth conflict.
if _SP_CLIENT_ID and _SP_CLIENT_SECRET:
    WORKSPACE_CLIENT = WorkspaceClient(config=Config(
        host=os.environ.get("DATABRICKS_HOST"),
        client_id=_SP_CLIENT_ID,
        client_secret=_SP_CLIENT_SECRET,
        auth_type="oauth-m2m",
    ))
else:
    WORKSPACE_CLIENT = WorkspaceClient()

# Ensure DATABRICKS_HOST is set
if not os.environ.get("DATABRICKS_HOST") and WORKSPACE_CLIENT.config.host:
    os.environ["DATABRICKS_HOST"] = WORKSPACE_CLIENT.config.host.rstrip("/")

# MLflow setup
if os.environ.get("DATABRICKS_HOST") and not os.environ.get("MLFLOW_TRACKING_URI"):
    os.environ["MLFLOW_TRACKING_URI"] = "databricks"

mlflow.set_registry_uri("databricks-uc")
MLFLOW_EXPERIMENT_ID = CONFIG["mlflow"]["experiment_id"]
if MLFLOW_EXPERIMENT_ID:
    try:
        mlflow.set_experiment(experiment_id=MLFLOW_EXPERIMENT_ID)
        logger.info(f"MLflow experiment set to {MLFLOW_EXPERIMENT_ID}")
    except Exception as e:
        logger.warning(f"Could not set MLflow experiment: {e}")

# Load system prompt
try:
    PROMPT_URI_AGENT = f"prompts:/{PROMPT_NAME}@production"
    SYSTEM_PROMPT = mlflow.genai.load_prompt(PROMPT_URI_AGENT)
except:
    SYSTEM_PROMPT = None
    logger.warning("Could not load system prompt from registry, using None")

# Lakebase Configuration
DISABLE_LAKEBASE = os.getenv("DISABLE_LAKEBASE", "false").lower() == "true"

if DISABLE_LAKEBASE:
    print("[CONFIG] Lakebase DISABLED via DISABLE_LAKEBASE env var")
    LAKEBASE_CONFIG = {"instance_name": "", "conn_host": ""}
else:
    LAKEBASE_CONFIG = {
        "instance_name": CONFIG["lakebase"]["instance_name"],
        "conn_host": CONFIG["lakebase"]["conn_host"],
        "conn_db_name": "databricks_postgres",
        "conn_ssl_mode": "require",
    }
    print(f"[CONFIG] Lakebase instance: {LAKEBASE_CONFIG['instance_name']}")
    print(f"[CONFIG] Lakebase host: {LAKEBASE_CONFIG['conn_host'] or '(not configured)'}")

########################################
# MCP PARALLEL EXECUTION
########################################

# Cache auth for per-thread MCP clients (reuse the same local SP creds)
_CACHED_HOST = os.environ.get("DATABRICKS_HOST")
_CACHED_TOKEN = os.environ.get("DATABRICKS_TOKEN")


def _execute_uc_function_mcp(tool_name: str, args: dict) -> str:
    """Execute a UC function using Databricks MCP with per-thread client isolation."""
    host = os.environ.get("DATABRICKS_HOST", "").rstrip("/")
    if not host:
        host = WORKSPACE_CLIENT.config.host.rstrip("/")
    mcp_server_url = f"{host}/api/2.0/mcp/functions/{CATALOG}/{SCHEMA}"

    mcp_tool_name = _resolve_tool_name(tool_name.replace(".", "__") if "." in tool_name else tool_name)

    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)

    try:
        # Create per-thread client — explicit auth_type avoids dual-auth
        # conflicts when the notebook runtime re-injects DATABRICKS_TOKEN.
        if _SP_CLIENT_ID and _SP_CLIENT_SECRET:
            thread_workspace_client = WorkspaceClient(config=Config(
                host=_CACHED_HOST,
                client_id=_SP_CLIENT_ID,
                client_secret=_SP_CLIENT_SECRET,
                auth_type="oauth-m2m",
            ))
        elif _CACHED_TOKEN:
            thread_workspace_client = WorkspaceClient(config=Config(
                host=_CACHED_HOST,
                token=_CACHED_TOKEN,
                auth_type="pat",
            ))
        else:
            thread_workspace_client = WorkspaceClient()

        mcp_client = DatabricksMCPClient(
            server_url=mcp_server_url,
            workspace_client=thread_workspace_client
        )

        result = mcp_client.call_tool(mcp_tool_name, args)

        if hasattr(result, 'content'):
            if isinstance(result.content, list):
                return "\n".join(
                    str(item.text if hasattr(item, 'text') else item)
                    for item in result.content
                )
            return str(result.content)
        return str(result)

    except BaseException as e:
        # Recursively unwrap TaskGroup / ExceptionGroup to surface the root cause
        real_error = e
        while hasattr(real_error, 'exceptions') and real_error.exceptions:
            real_error = real_error.exceptions[0]
        logger.error(f"MCP call failed for {tool_name}: {type(real_error).__name__}: {real_error}")
        raise real_error from None
    finally:
        loop.close()


########################################
# LAKEBASE CONNECTION POOLING
########################################

class CredentialConnection(psycopg.Connection):
    """Custom connection class with OAuth token caching."""

    workspace_client = None
    instance_name = None
    _cached_credential = None
    _cache_timestamp = None
    _cache_duration = 3000  # 50 minutes
    _cache_lock = Lock()

    @classmethod
    def connect(cls, conninfo='', **kwargs):
        if cls.workspace_client is None or cls.instance_name is None:
            raise ValueError("workspace_client and instance_name must be set")
        kwargs['password'] = cls._get_cached_credential()
        return super().connect(conninfo, **kwargs)

    @classmethod
    def _get_cached_credential(cls):
        with cls._cache_lock:
            current_time = time.time()
            if (cls._cached_credential is not None and
                cls._cache_timestamp is not None and
                current_time - cls._cache_timestamp < cls._cache_duration):
                return cls._cached_credential

            credential = cls.workspace_client.database.generate_database_credential(
                request_id=str(uuid.uuid4()),
                instance_names=[cls.instance_name]
            )
            cls._cached_credential = credential.token
            cls._cache_timestamp = current_time
            return cls._cached_credential


########################################
# GENIE CONFIGURATION
########################################

class GenieConfig(BaseModel):
    space_id: str
    name: str
    description: str = ""
    max_retries: int = 2


GENIE_CONFIG = GenieConfig(
    space_id=GENIE_SPACE_ID,
    name=GENIE_NAME,
    description="""This agent is a FALLBACK for when UC functions cannot answer the question.
The Genie space contains comprehensive baseball Statcast pitch-level data including:
- Pitch characteristics (velocity, spin, movement, release point)
- Batter outcomes (launch speed, angle, hit distance, wOBA)
- Player and team information
- Historical matchup data
Use this for flexible, novel queries that predefined UC functions cannot handle.""",
    max_retries=2,
)


########################################
# AGENT STATE
########################################

class AgentState(TypedDict):
    """State for the combined agent with memory and Genie fallback."""
    messages: Annotated[Sequence[BaseMessage], add_messages]
    original_query: str
    uc_response: str | None
    uc_answered_parts: str | None
    unanswered_parts: str | None
    uc_sufficient: bool
    genie_response: str | None
    genie_error: str | None
    final_response: str | None
    tool_calls_made: list[dict[str, Any]]
    custom_inputs: Optional[dict[str, Any]]
    custom_outputs: Optional[dict[str, Any]]


########################################
# LOAD UC TOOLS
########################################

tools = []
if UC_TOOL_NAMES:
    try:
        print(f"[STARTUP] Loading {len(UC_TOOL_NAMES)} UC tools...")
        uc_toolkit = UCFunctionToolkit(function_names=UC_TOOL_NAMES)
        tools.extend(uc_toolkit.tools)
        print(f"[STARTUP] UC tools loaded successfully: {len(tools)} tools")
    except Exception as e:
        print(f"[STARTUP ERROR] Failed to load UC toolkit: {e}")
        logger.error(f"Failed to load UC toolkit: {e}")
else:
    print("[STARTUP] No UC tools configured")


# Build lookup for resolving potentially truncated tool names from LLM
_TOOL_NAME_SUFFIX_MAP = {}
for _tn in UC_TOOL_NAMES:
    # Full dotted name -> underscored MCP name
    _mcp_name = _tn.replace(".", "__")
    _TOOL_NAME_SUFFIX_MAP[_mcp_name] = _mcp_name
    # Also map by just the function name (last segment)
    _func_name = _tn.split(".")[-1]
    _TOOL_NAME_SUFFIX_MAP[_func_name] = _mcp_name


def _resolve_tool_name(name: str) -> str:
    """Resolve a potentially truncated tool name to the correct full MCP name.
    
    The LLM sometimes truncates long catalog names (e.g. 'cmegdemos_catalog' -> 'mos_catalog').
    This resolves by matching the function suffix.
    """
    if name in _TOOL_NAME_SUFFIX_MAP:
        return _TOOL_NAME_SUFFIX_MAP[name]
    # Try matching by suffix (function name after last __)
    parts = name.split("__")
    func_suffix = parts[-1] if parts else name
    if func_suffix in _TOOL_NAME_SUFFIX_MAP:
        resolved = _TOOL_NAME_SUFFIX_MAP[func_suffix]
        logger.info(f"Resolved truncated tool name '{name}' -> '{resolved}'")
        return resolved
    # Try endswith matching
    for full_name in _TOOL_NAME_SUFFIX_MAP.values():
        if full_name.endswith(func_suffix):
            logger.info(f"Resolved truncated tool name '{name}' -> '{full_name}'")
            return full_name
    return name


########################################
# MAIN AGENT CLASS
########################################

class CombinedLangGraphAgent(ResponsesAgent):
    """
    Combined agent with:
    - UC-first, Genie-fallback architecture
    - Parallel MCP tool execution
    - Lakebase PostgreSQL memory
    - Partial answer detection
    """

    def __init__(self, lakebase_config: dict[str, Any], genie_config: GenieConfig):
        self.lakebase_config = lakebase_config
        self.genie_config = genie_config
        self.workspace_client = WORKSPACE_CLIENT

        # LLM setup
        self.model = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME)
        self.system_prompt = SYSTEM_PROMPT
        self.model_with_tools = self.model.bind_tools(tools) if tools else self.model

        # Genie agent
        self.genie_agent = GenieAgent(
            genie_space_id=genie_config.space_id,
            genie_agent_name=genie_config.name,
            description=genie_config.description,
        )

        # Connection pool settings
        self.pool_min_size = int(os.getenv("DB_POOL_MIN_SIZE", "1"))
        self.pool_max_size = int(os.getenv("DB_POOL_MAX_SIZE", "10"))
        self.pool_timeout = float(os.getenv("DB_POOL_TIMEOUT", "30.0"))

        cache_duration_minutes = int(os.getenv("DB_TOKEN_CACHE_MINUTES", "50"))
        CredentialConnection._cache_duration = cache_duration_minutes * 60

        # Only create pool if Lakebase is configured
        self._connection_pool = None
        if self.lakebase_config.get("conn_host"):
            try:
                self._connection_pool = self._create_rotating_pool()
            except Exception as e:
                logger.warning(f"Failed to create Lakebase connection pool: {e}")

        mlflow.langchain.autolog()

    def _get_username(self) -> str:
        try:
            sp = self.workspace_client.current_service_principal.me()
            return sp.application_id
        except Exception:
            user = self.workspace_client.current_user.me()
            return user.user_name

    def _create_rotating_pool(self) -> ConnectionPool:
        CredentialConnection.workspace_client = self.workspace_client
        CredentialConnection.instance_name = self.lakebase_config["instance_name"]

        username = self._get_username()
        host = self.lakebase_config["conn_host"]
        database = self.lakebase_config.get("conn_db_name", "databricks_postgres")

        pool = ConnectionPool(
            conninfo=f"dbname={database} user={username} host={host} sslmode=require options='-c search_path=checkpoint'",
            connection_class=CredentialConnection,
            min_size=self.pool_min_size,
            max_size=self.pool_max_size,
            timeout=self.pool_timeout,
            open=True,
            check=ConnectionPool.check_connection,
            kwargs={
                "autocommit": True,
                "row_factory": dict_row,
                "keepalives": 1,
                "keepalives_idle": 60,
                "keepalives_interval": 10,
                "keepalives_count": 5,
            }
        )

        # Test connection
        with pool.connection() as conn:
            with conn.cursor() as cursor:
                cursor.execute("SELECT 1")

        return pool

    @contextmanager
    def get_connection(self):
        """Get a validated connection from the pool with retry logic."""
        if not self._connection_pool:
            yield None
            return

        max_retries = 3
        retry_count = 0

        while retry_count < max_retries:
            try:
                with self._connection_pool.connection() as conn:
                    with conn.cursor() as cursor:
                        cursor.execute("SELECT 1")
                    yield conn
                    return
            except (psycopg.OperationalError, psycopg.InterfaceError) as e:
                retry_count += 1
                if retry_count >= max_retries:
                    self._connection_pool = self._create_rotating_pool()
                    with self._connection_pool.connection() as conn:
                        yield conn
                        return
                time.sleep(0.5 * retry_count)

    def _parallel_tool_node(self, state: AgentState) -> dict:
        """Execute all tool calls in parallel using MCP with MLflow tracing."""
        messages = state["messages"]
        last_message = messages[-1] if messages else None

        if not isinstance(last_message, AIMessage) or not last_message.tool_calls:
            return {"messages": [], "tool_calls_made": []}

        tool_calls = last_message.tool_calls
        start_time = time.time()

        with ThreadPoolExecutor(max_workers=min(len(tool_calls), 10)) as executor:
            futures = {}
            tool_start_times = {}
            for tc in tool_calls:
                tool_start_times[tc["id"]] = time.time()
                future = executor.submit(_execute_uc_function_mcp, tc["name"], tc["args"])
                futures[future] = {"id": tc["id"], "name": tc["name"], "args": tc["args"]}

            tool_results = []
            for future in as_completed(futures):
                tool_info = futures[future]
                elapsed_tool = time.time() - tool_start_times[tool_info["id"]]
                try:
                    result = future.result()
                    error = False
                except Exception as e:
                    result = f"Error: {str(e)}"
                    error = True

                tool_results.append({
                    "id": tool_info["id"],
                    "name": tool_info["name"],
                    "args": tool_info["args"],
                    "result": result,
                    "duration": elapsed_tool,
                    "error": error
                })

        elapsed = time.time() - start_time
        logger.info(f"Parallel tool execution: {len(tool_results)} tools in {elapsed:.2f}s")

        # MLflow tracing for parallel tool execution
        with mlflow.start_span(
            name=f"parallel_tools ({len(tool_results)} tools, {elapsed:.2f}s)",
            span_type=SpanType.TOOL
        ) as parent_span:
            parent_span.set_inputs({
                "num_tools": len(tool_results),
                "execution_mode": "parallel_mcp",
                "tool_names": [tr["name"] for tr in tool_results]
            })
            parent_span.set_attribute("wall_time_seconds", elapsed)

            for tr in tool_results:
                with mlflow.start_span(
                    name=f"{tr['name']} ({tr['duration']:.2f}s)",
                    span_type=SpanType.TOOL
                ) as span:
                    span.set_inputs({"tool_name": tr["name"], "args": tr["args"]})
                    span.set_outputs({
                        "execution_time_seconds": tr["duration"],
                        "result_preview": tr["result"][:500] if tr["result"] else "",
                        "error": tr["error"]
                    })

            parent_span.set_outputs({
                "completed": len(tool_results),
                "total_wall_time_seconds": elapsed,
            })

        tool_messages = [
            ToolMessage(content=tr["result"], tool_call_id=tr["id"])
            for tr in tool_results
        ]

        return {"messages": tool_messages, "tool_calls_made": tool_results}

    def _evaluate_sufficiency(self, state: AgentState) -> dict:
        """Evaluate if UC response is sufficient, partial, or not answered."""
        original_query = state.get("original_query", "")
        uc_response = state.get("uc_response", "")

        eval_prompt = f"""You are evaluating whether a response adequately answers a user's question.
The question may have MULTIPLE parts. Evaluate EACH part separately.

Original Question: {original_query}

Response from UC Functions Agent: {uc_response}

Analyze the response and categorize it:
1. FULLY_ANSWERED - The response completely answers ALL parts with relevant data
2. PARTIALLY_ANSWERED - The response answers SOME parts but not others
3. NOT_ANSWERED - The response doesn't answer the question (just "I cannot answer" or errors)

If PARTIALLY_ANSWERED, identify what parts were NOT answered.

Respond in this EXACT format:
STATUS: [FULLY_ANSWERED or PARTIALLY_ANSWERED or NOT_ANSWERED]
ANSWERED_PARTS: [Brief summary of what was answered, or "None"]
UNANSWERED_PARTS: [Specific questions that still need answers, or "None"]

Your evaluation:"""

        eval_result = self.model.invoke([HumanMessage(content=eval_prompt)])
        eval_content = eval_result.content.upper()

        is_sufficient = "FULLY_ANSWERED" in eval_content
        is_partial = "PARTIALLY_ANSWERED" in eval_content

        answered_parts = None
        unanswered_parts = None

        try:
            lines = eval_result.content.split('\n')
            for line in lines:
                if line.upper().startswith("ANSWERED_PARTS:"):
                    answered_parts = line.split(":", 1)[1].strip()
                    if answered_parts.upper() == "NONE":
                        answered_parts = None
                elif line.upper().startswith("UNANSWERED_PARTS:"):
                    unanswered_parts = line.split(":", 1)[1].strip()
                    if unanswered_parts.upper() == "NONE":
                        unanswered_parts = None
        except Exception as e:
            logger.warning(f"Failed to parse sufficiency evaluation: {e}")

        if is_partial and unanswered_parts:
            status = "PARTIALLY_ANSWERED"
            result = {
                "uc_sufficient": False,
                "uc_answered_parts": answered_parts,
                "unanswered_parts": unanswered_parts,
            }
        elif is_sufficient:
            status = "FULLY_ANSWERED"
            result = {"uc_sufficient": True, "uc_answered_parts": answered_parts, "unanswered_parts": None}
        else:
            status = "NOT_ANSWERED"
            result = {"uc_sufficient": False, "uc_answered_parts": None, "unanswered_parts": original_query}

        # MLflow tracing
        with mlflow.start_span(name=f"sufficiency_eval ({status})", span_type=SpanType.LLM) as span:
            span.set_inputs({"original_query": original_query})
            span.set_outputs({"status": status, "will_fallback_to_genie": not result["uc_sufficient"]})

        return result

    def _genie_fallback_node(self, state: AgentState) -> dict:
        """Fall back to Genie for unanswered parts."""
        original_query = state.get("original_query", "")
        unanswered_parts = state.get("unanswered_parts", "")

        if unanswered_parts and unanswered_parts != original_query:
            genie_query = f"""I need help answering a specific part of a question about baseball data.
The user's full question was: {original_query}

I was able to answer part of it, but I need your help with:
{unanswered_parts}

Please focus on answering ONLY the part I couldn't answer."""
        else:
            genie_query = f"""I need help answering this question about baseball data.
Question: {original_query}
Please use your data access capabilities to answer this question."""

        genie_messages = [HumanMessage(content=genie_query)]

        genie_response = ""
        genie_error = None
        start_time = time.time()

        for attempt in range(self.genie_config.max_retries + 1):
            try:
                logger.info(f"Genie fallback attempt {attempt + 1}")
                result = self.genie_agent.invoke({"messages": genie_messages})

                for msg in reversed(result.get("messages", [])):
                    if hasattr(msg, "content") and msg.content:
                        genie_response = msg.content
                        break

                if genie_response:
                    break

            except Exception as e:
                genie_error = f"Genie error: {str(e)}"
                logger.warning(f"Genie error on attempt {attempt + 1}: {e}")
                if attempt < self.genie_config.max_retries:
                    time.sleep(2 ** attempt)

        elapsed = time.time() - start_time

        with mlflow.start_span(name=f"genie_fallback ({elapsed:.2f}s)", span_type=SpanType.CHAIN) as span:
            span.set_inputs({"genie_space_id": self.genie_config.space_id, "original_query": original_query})
            span.set_outputs({"success": bool(genie_response), "duration_seconds": elapsed})

        return {"genie_response": genie_response, "genie_error": genie_error}

    def _synthesize_response(self, state: AgentState) -> dict:
        """Synthesize final response from UC and Genie outputs."""
        uc_response = state.get("uc_response", "")
        genie_response = state.get("genie_response", "")
        genie_error = state.get("genie_error")
        uc_sufficient = state.get("uc_sufficient", False)
        uc_answered_parts = state.get("uc_answered_parts")
        unanswered_parts = state.get("unanswered_parts")
        original_query = state.get("original_query", "")

        if uc_sufficient:
            final_response = uc_response
        elif genie_response:
            if uc_answered_parts and unanswered_parts != original_query:
                synthesis_prompt = f"""Combine two partial responses into a unified answer.

CURRENT QUESTION (answer ONLY this): {original_query}

PART 1 - From UC Functions:
{uc_response}

PART 2 - From Genie:
{genie_response}

IMPORTANT: Only include information that DIRECTLY answers the current question above.
Do NOT include information from prior conversation turns unless explicitly referenced.
Create a UNIFIED response that presents all relevant information clearly without mentioning different systems."""
            else:
                synthesis_prompt = f"""Synthesize a response based on available information.

CURRENT QUESTION (answer ONLY this): {original_query}

Available Information:
- UC Functions: {uc_response}
- Genie: {genie_response}

IMPORTANT: Only include information that DIRECTLY answers the current question.
Provide a clear, focused response."""

            synthesis_result = self.model.invoke([HumanMessage(content=synthesis_prompt)])
            final_response = synthesis_result.content

        elif genie_error:
            if uc_response and uc_answered_parts:
                final_response = f"""{uc_response}

---
Note: I answered part of your question above but couldn't get additional information for: {unanswered_parts}
Please try asking about that specific part again."""
            else:
                final_response = f"I couldn't fully answer your question. Error: {genie_error}"
        else:
            final_response = uc_response or "I wasn't able to find an answer."

        return {"final_response": final_response, "messages": [AIMessage(content=final_response)]}

    def _create_graph(self, checkpointer=None):
        """Create the LangGraph workflow with UC-first, Genie-fallback pattern."""

        def should_continue_tools(state: AgentState):
            messages = state["messages"]
            last_message = messages[-1] if messages else None
            if isinstance(last_message, AIMessage) and last_message.tool_calls:
                return "tools"
            return "evaluate"

        def route_after_evaluation(state: AgentState) -> Literal["genie_fallback", "synthesize"]:
            if state.get("uc_sufficient", False):
                return "synthesize"
            return "genie_fallback"

        # Preprocessor with system prompt
        if self.system_prompt:
            prompt_text = self.system_prompt.format() if hasattr(self.system_prompt, 'format') else str(self.system_prompt)
            preprocessor = RunnableLambda(
                lambda state: [{"role": "system", "content": prompt_text}] + list(state["messages"])
            )
        else:
            preprocessor = RunnableLambda(lambda state: list(state["messages"]))

        model_runnable = preprocessor | self.model_with_tools

        def call_model(state: AgentState, config: RunnableConfig):
            response = model_runnable.invoke(state, config)
            original_query = state.get("original_query", "")
            if not original_query:
                for msg in reversed(state["messages"]):
                    if isinstance(msg, HumanMessage):
                        original_query = msg.content
                        break
            return {"messages": [response], "original_query": original_query}

        def extract_uc_response(state: AgentState):
            messages = state["messages"]
            uc_response = ""
            for msg in reversed(messages):
                if isinstance(msg, AIMessage) and msg.content and not msg.tool_calls:
                    uc_response = msg.content
                    break
            return {"uc_response": uc_response}

        workflow = StateGraph(AgentState)

        # Add nodes
        workflow.add_node("agent", RunnableLambda(call_model))
        workflow.add_node("tools", self._parallel_tool_node)
        workflow.add_node("extract_response", extract_uc_response)
        workflow.add_node("evaluate", self._evaluate_sufficiency)
        workflow.add_node("genie_fallback", self._genie_fallback_node)
        workflow.add_node("synthesize", self._synthesize_response)

        # Add edges
        workflow.set_entry_point("agent")
        workflow.add_conditional_edges(
            "agent",
            should_continue_tools,
            {"tools": "tools", "evaluate": "extract_response"}
        )
        workflow.add_edge("tools", "agent")
        workflow.add_edge("extract_response", "evaluate")
        workflow.add_conditional_edges(
            "evaluate",
            route_after_evaluation,
            {"genie_fallback": "genie_fallback", "synthesize": "synthesize"}
        )
        workflow.add_edge("genie_fallback", "synthesize")
        workflow.add_edge("synthesize", END)

        return workflow.compile(checkpointer=checkpointer)

    def _get_or_create_thread_id(self, request: ResponsesAgentRequest) -> str:
        ci = dict(request.custom_inputs or {})
        if "thread_id" in ci:
            return ci["thread_id"]
        if request.context and getattr(request.context, "conversation_id", None):
            return request.context.conversation_id
        return str(uuid.uuid4())

    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        thread_id = self._get_or_create_thread_id(request)
        ci = dict(request.custom_inputs or {})
        ci["thread_id"] = thread_id
        request.custom_inputs = ci

        outputs = [
            event.item
            for event in self.predict_stream(request)
            if event.type == "response.output_item.done"
        ]
        return ResponsesAgentResponse(output=outputs, custom_outputs={"thread_id": thread_id})

    def predict_stream(
        self,
        request: ResponsesAgentRequest,
    ) -> Generator[ResponsesAgentStreamEvent, None, None]:
        """Streaming prediction with PostgreSQL checkpointing and MLflow tracing."""
        thread_id = self._get_or_create_thread_id(request)

        ci = dict(request.custom_inputs or {})
        ci["thread_id"] = thread_id
        request.custom_inputs = ci

        cc_msgs = self.prep_msgs_for_cc_llm([i.model_dump() for i in request.input])
        checkpoint_config = {"configurable": {"thread_id": thread_id}}

        # Convert to LangChain messages
        lc_messages = []
        for msg in cc_msgs:
            role = msg.get("role", "user")
            content = msg.get("content", "")
            if role == "user":
                lc_messages.append(HumanMessage(content=content))
            elif role == "assistant":
                lc_messages.append(AIMessage(content=content))

        # MLflow tracing for Lakebase session
        with mlflow.start_span(name="lakebase_session", span_type=SpanType.RETRIEVER) as db_span:
            db_span.set_inputs({
                "thread_id": thread_id,
                "lakebase_instance": self.lakebase_config.get("instance_name", "unknown"),
                "memory_enabled": bool(self._connection_pool)
            })

            with self.get_connection() as conn:
                checkpointer = PostgresSaver(conn) if conn else None
                graph = self._create_graph(checkpointer=checkpointer)

                node_count = 0
                final_response = None
                emitted_tool_calls = set()
                tracked_unanswered_parts = ""

                for event in graph.stream(
                    {"messages": lc_messages},
                    checkpoint_config,
                    stream_mode=["updates", "messages"]
                ):
                    if event[0] == "updates":
                        node_count += 1
                        node_name = list(event[1].keys())[0] if event[1] else ""
                        node_data = event[1].get(node_name, {})

                        # Emit function_call events when tools node executes
                        if node_name == "tools" and "tool_calls_made" in node_data:
                            for tc in node_data["tool_calls_made"]:
                                if tc["id"] not in emitted_tool_calls:
                                    emitted_tool_calls.add(tc["id"])
                                    yield ResponsesAgentStreamEvent(
                                        type="response.output_item.done",
                                        item=self.create_function_call_item(
                                            id=str(uuid.uuid4()),
                                            call_id=tc["id"],
                                            name=tc["name"],
                                            arguments=json.dumps(tc["args"]) if isinstance(tc["args"], dict) else tc["args"],
                                        ),
                                        custom_outputs={"thread_id": thread_id}
                                    )
                                    yield ResponsesAgentStreamEvent(
                                        type="response.output_item.done",
                                        item=self.create_function_call_output_item(
                                            call_id=tc["id"],
                                            output=tc["result"][:2000] if tc["result"] else "",
                                        ),
                                        custom_outputs={"thread_id": thread_id}
                                    )

                        if node_name == "evaluate" and "unanswered_parts" in node_data:
                            tracked_unanswered_parts = node_data.get("unanswered_parts", "")

                        # Emit Genie fallback event
                        if node_name == "genie_fallback":
                            genie_call_id = f"genie-{uuid.uuid4()}"
                            genie_response = node_data.get("genie_response", "")
                            query_display = tracked_unanswered_parts if tracked_unanswered_parts else "Genie fallback query"
                            yield ResponsesAgentStreamEvent(
                                type="response.output_item.done",
                                item=self.create_function_call_item(
                                    id=str(uuid.uuid4()),
                                    call_id=genie_call_id,
                                    name="genie_space_query",
                                    arguments=json.dumps({"query": query_display[:200]}),
                                ),
                                custom_outputs={"thread_id": thread_id}
                            )
                            yield ResponsesAgentStreamEvent(
                                type="response.output_item.done",
                                item=self.create_function_call_output_item(
                                    call_id=genie_call_id,
                                    output=genie_response[:500] if genie_response else "(queried Genie space)",
                                ),
                                custom_outputs={"thread_id": thread_id}
                            )

                        # Track final response from synthesize node
                        if node_name == "synthesize" and "final_response" in node_data:
                            final_response = node_data["final_response"]
                        elif node_name == "synthesize" and "messages" in node_data:
                            for msg in node_data["messages"]:
                                if isinstance(msg, AIMessage) and msg.content:
                                    final_response = msg.content

                # Emit final synthesized response
                if final_response:
                    yield ResponsesAgentStreamEvent(
                        type="response.output_item.done",
                        item=self.create_text_output_item(
                            text=final_response,
                            id=str(uuid.uuid4()),
                        ),
                        custom_outputs={"thread_id": thread_id}
                    )

                db_span.set_outputs({
                    "status": "completed",
                    "thread_id": thread_id,
                    "nodes_executed": node_count,
                    "tool_calls_emitted": len(emitted_tool_calls)
                })


########################################
# INSTANTIATE AGENT
########################################

AGENT = CombinedLangGraphAgent(LAKEBASE_CONFIG, GENIE_CONFIG)
mlflow.models.set_model(AGENT)

In [ ]:
dbutils.library.restartPython()

## Test the Agent

### Expected Behavior
1. **UC Functions First**: The agent always tries to answer using Unity Catalog functions first
2. **Sufficiency Check**: LLM evaluates if the UC response is FULLY, PARTIALLY, or NOT answered
3. **Partial Answer Detection**: If UC answered some parts but not others, only unanswered parts go to Genie
4. **Genie Fallback**: For unanswered parts, Genie is automatically invoked
5. **Response Synthesis**: The final response intelligently combines both sources

In [ ]:
# Reload config after Python restart
import json
from pathlib import Path
import mlflow

CONFIG = json.loads(Path("config/atbat_assistant.json").read_text())

# Re-extract all configuration variables (needed for logging/deployment cells)
PROMPT_NAME = CONFIG["prompt_registry"]["prompt_name"]
LLM_ENDPOINT_NAME = CONFIG["llm"]["endpoint_name"]
UC_MODEL_NAME = CONFIG["model"]["uc_model_name"]
UC_TOOL_NAMES = CONFIG["tools"]["uc_tool_names"]
CATALOG = CONFIG["workspace"]["catalog"]
SCHEMA = CONFIG["workspace"]["schema"]
GENIE_SPACE_ID = CONFIG["genie"]["space_id"]
GENIE_NAME = CONFIG["genie"]["name"]
LAKEBASE_INSTANCE = CONFIG["lakebase"]["instance_name"]

# Set MLflow experiment
EXPERIMENT_ID = CONFIG["mlflow"]["experiment_id"]
mlflow.set_experiment(experiment_id=EXPERIMENT_ID)

In [ ]:
from agent import AGENT
# Test 1: UC Functions Only
print("=== Test 1: UC Functions Only ===")
result = AGENT.predict({"input": [{"role": "user", "content": "How will Kyle Freeland pitch to Freddie Freeman?"}]})
print(f"Thread ID: {result.custom_outputs.get('thread_id')}")
print(f"Response preview: {result.output[-1].content[0]['text'][:500]}...")

In [ ]:

# Test 2: UC Functions AND Genie
print("=== Test 2: UC Functions + Genie Fallback ===")
result = AGENT.predict({"input": [{"role": "user", "content": "How will Blake Snell pitch to Mookie Betts? What was the full pitch distribution for the Rockies in 2025 including the average spin rate and velocity by pitch?"}]})
print(f"Thread ID: {result.custom_outputs.get('thread_id')}")
print(f"Response preview: {result.output[-1].content[0]['text'][:500]}...")

In [ ]:
# Test 3: Streaming
print("=== Test 3: Streaming ===")
for chunk in AGENT.predict_stream(
    {"input": [{"role": "user", "content": "How does Yu Darvish pitch to lefties with runners on 2nd?"}]}
):
    print(chunk.model_dump(exclude_none=True))

In [ ]:
# Test 4: Memory test (uses same thread_id)
print("=== Test 4: Memory Test ===")
thread_id = "atbat-memory-test-001"

result1 = AGENT.predict({
    "input": [{"role": "user", "content": "The secret code is HammerinHank44"}],
    "custom_inputs": {"thread_id": thread_id}
})

result2 = AGENT.predict({
    "input": [{"role": "user", "content": "What was the secret code I provided?"}],
    "custom_inputs": {"thread_id": thread_id}
})
print(f"Memory response: {result2.output[-1].content[0]['text'][:300]}")

## Log and Deploy the Agent

In [ ]:
# Determine Databricks resources for automatic auth passthrough at deployment time
# Docs: https://docs.databricks.com/aws/en/generative-ai/agent-framework/agent-authentication
#
# IMPORTANT: You must declare ALL downstream resources the agent touches.
# The auto-provisioned SP only gets least-privilege access to declared resources.
# Per the docs: "if you log a Genie Space, you must also log its tables,
# SQL Warehouses, and Unity Catalog functions."
from mlflow.models.resources import (
    DatabricksFunction,
    DatabricksGenieSpace,
    DatabricksLakebase,
    DatabricksServingEndpoint,
    DatabricksSQLWarehouse,
    DatabricksTable,
    DatabricksVectorSearchIndex,
)
from pkg_resources import get_distribution

warehouse_id = CONFIG["genie"]["warehouse_id"]
uc_schema = f"{CATALOG}.{SCHEMA}"

resources = [DatabricksServingEndpoint(endpoint_name=LLM_ENDPOINT_NAME)]

# Add UC functions from config
for tool_name in UC_TOOL_NAMES:
    resources.append(DatabricksFunction(function_name=tool_name))

# Vector Search indices
BATTER_INDEX_NAME = CONFIG["vector_search"]["batter_index_name"]
PITCHER_INDEX_NAME = CONFIG["vector_search"]["pitcher_index_name"]
resources.append(DatabricksVectorSearchIndex(index_name=BATTER_INDEX_NAME))
resources.append(DatabricksVectorSearchIndex(index_name=PITCHER_INDEX_NAME))

# Genie space + SQL warehouse (required for Genie to execute queries)
resources.append(DatabricksSQLWarehouse(warehouse_id=warehouse_id))
resources.append(DatabricksGenieSpace(genie_space_id=GENIE_SPACE_ID))

# Tables the Genie space queries — REQUIRED per docs.
# The auto-provisioned SP needs SELECT on every table the Genie space touches.
# Update this list to match the tables configured in your Genie space.
GENIE_TABLES = CONFIG["genie"]["tables"]
for table in GENIE_TABLES:
    resources.append(DatabricksTable(table_name=f"{uc_schema}.{table}"))

# Lakebase instance (for LangGraph conversation memory)
# Grants the auto-provisioned SP databricks_superuser on the Lakebase instance
resources.append(DatabricksLakebase(database_instance_name=LAKEBASE_INSTANCE))

input_example = {
    "input": [
        {"role": "user", "content": "What are similar batters to Kyle Schwarber?"}
    ]
}

with mlflow.start_run():
    logged_agent_info = mlflow.pyfunc.log_model(
        name="agent",
        python_model="agent.py",
        input_example=input_example,
        resources=resources,
        pip_requirements=[
            "databricks-openai",
            "backoff",
            f"databricks-connect=={get_distribution('databricks-connect').version}",
            f"databricks-langchain=={get_distribution('databricks-langchain').version}",
            f"langgraph=={get_distribution('langgraph').version}",
            "langgraph-checkpoint-postgres",
            "psycopg[binary,pool]",
            "databricks-mcp",
        ],
    )
    print(f"Logged model: {logged_agent_info.model_uri}")

In [ ]:
# Register to Unity Catalog
mlflow.set_registry_uri("databricks-uc")

uc_registered_model_info = mlflow.register_model(
    model_uri=logged_agent_info.model_uri,
    name=UC_MODEL_NAME
)
print(f"Registered model: {UC_MODEL_NAME} version {uc_registered_model_info.version}")

In [ ]:
# Deploy the agent
from databricks import agents

# Build environment variables - package the config so deployed agent can load it
config_payload = Path("config/atbat_assistant.json").read_text()
environment_vars = {
    "ATBAT_ASSISTANT_CONFIG_JSON": config_payload,
    "MLFLOW_TRACKING_URI": "databricks",
}

# Build authentication environment variables
auth_config = CONFIG["prompt_registry_auth"]
if auth_config.get("databricks_host"):
    if auth_config.get("use_oauth", True):
        environment_vars.update({
            "DATABRICKS_HOST": auth_config["databricks_host"],
            "DATABRICKS_CLIENT_ID": f"{{{{secrets/{auth_config['secret_scope_name']}/{auth_config['oauth_client_id_key']}}}}}",
            "DATABRICKS_CLIENT_SECRET": f"{{{{secrets/{auth_config['secret_scope_name']}/{auth_config['oauth_client_secret_key']}}}}}",
        })
        print(f"Using OAuth authentication (scope: {auth_config['secret_scope_name']})")
        print(f"MLflow tracking URI set to 'databricks' - traces will go to experiment {CONFIG['mlflow']['experiment_id']}")
    else:
        environment_vars.update({
            "DATABRICKS_HOST": auth_config["databricks_host"],
            "DATABRICKS_TOKEN": f"{{{{secrets/{auth_config['secret_scope_name']}/{auth_config.get('pat_key', 'pat')}}}}}",
        })
        print(f"Using PAT authentication (scope: {auth_config['secret_scope_name']})")
else:
    print("WARNING: DATABRICKS_HOST not configured - Prompt Registry and MLflow tracking may not work")

deployment = agents.deploy(
    UC_MODEL_NAME,
    uc_registered_model_info.version,
    environment_vars=environment_vars,
)

# Print the serving endpoint name for app configuration
endpoint_name = f"agents_{UC_MODEL_NAME.replace('.', '-')}"
print(f"\nServing endpoint: {endpoint_name}")
print(f"Set this in your Databricks App config:")
print(f"  UC_MODEL_NAME={UC_MODEL_NAME}")
print(f"  -- or --")
print(f"  SERVING_ENDPOINT_NAME={endpoint_name}")

In [ ]:
# Grant CAN_QUERY on the serving endpoint to all workspace users.
# This ensures Databricks Apps (which run as their own service principal)
# can query the endpoint without manual permission setup.
#
# ⚠️  SECURITY NOTE: This grants ALL workspace users permission to query
# this endpoint. If you want to restrict access to specific users or groups,
# skip this cell and configure permissions manually in the serving endpoint UI.

endpoint_name = f"agents_{UC_MODEL_NAME.replace('.', '-')}"
endpoint_info = w.serving_endpoints.get(endpoint_name)

w.api_client.do(
    "PUT",
    f"/api/2.0/permissions/serving-endpoints/{endpoint_info.id}",
    body={
        "access_control_list": [
            {
                "group_name": "users",
                "permission_level": "CAN_QUERY",
            }
        ]
    },
)
print(f"Granted CAN_QUERY on '{endpoint_name}' to all workspace users")